In [243]:
# Python-native
import os
import io
import json
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import seaborn as sns

# Statistics
import scipy.stats as stats
from scipy.stats import chi2_contingency, brunnermunzel, gaussian_kde
import statsmodels.stats as sms
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import zt_ind_solve_power, GofChisquarePower

# Excel and report related
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image
from PIL import Image as PILImage

In [244]:
# Path of A/B test result files
path = Path(r'D:\High-usage\Data Science\Data Analyst Path\Projects\Analytical Projects\Jooble\AB Test Results\Data Prepared to Stat Testing')

files = list(path.glob('*.csv'))
if not files:
        raise FileNotFoundError("No CSV files found in the directory.")

# Sorting files by date suffix (e.g. "jooble_ml_ab_test_data_20250804") and creation date
sorted_dated_files = sorted(
        files, 
        key=lambda f: (datetime.strptime(f.stem.split('_')[-1], '%Y%m%d'), datetime.fromtimestamp(f.stat().st_birthtime)), 
        reverse=True
)

# Fetching the most recent file path
latest_file = sorted_dated_files[0]

# Reading the latest A/B test result file
ab_test = pd.read_csv(latest_file)

In [245]:
# Statistical method functions (selected based on metric data type)

def z_test_prop(variant_metric: pd.DataFrame) -> dict:
    """Z-test for proportions with a Confidence Interval, Effect Size (Cohen's h) and Power Analysis."""
        
    metric_col_name = variant_metric.iloc[:, 1].name

    # Calculating the number of successes and observations for each variant
    sizes = variant_metric.groupby('variant').agg(
        count=(metric_col_name, 'sum'),
        nobs=(metric_col_name, 'size')
    )

    # Reorder so 'control' is first, regardless of treatment group name (e.g. 'treatment' vs. 'experiment')
    # This helps to ensure consistent reporting and interpretation
    indexes = sizes.index.tolist()
    indexes.remove('control')
    new_order = ['control'] + indexes
    sizes = sizes.reindex(new_order)

    # Extracting counts and number of observations
    count = sizes['count'].to_list()
    nobs = sizes['nobs'].to_list()

    # Performing the z-test for proportions
    stat, pval = proportions_ztest(count, nobs, alternative='two-sided', prop_var=False)

    # Calculating the confidence interval for the difference in proportions
    ci_low, ci_upp = confint_proportions_2indep(
        count[1], nobs[1], count[0], nobs[0], method='agresti-caffo'
    )
    
    # Power analysis for the z-test
    prop1 = count[0] / nobs[0] # control proportion
    prop2 = count[1] / nobs[1] # treatment proportion
    nobs1 = nobs[0]
    nobs2 = nobs[1]
    ratio = nobs2 / nobs1 # treatment to control ratio
    alpha = 0.05
    
    # Calculate effect size (Cohen’s h)
    # Cohen’s h measures the absolute difference between two proportions on an arcsine scale, not the relative difference; it's independent of sample size
    # The larger the absolute difference between the proportions, the larger the effect size
        # h ≈ 0.20 → Small effect (in practical terms)
        # h ≈ 0.50 → Medium effect (in practical terms)
        # h ≈ 0.80 → Large effect (in practical terms)
    effect_size = proportion_effectsize(prop2, prop1)

    # Calculate power for given effect size, sample, and alpha
    power = zt_ind_solve_power(effect_size=effect_size, power=None, nobs1=nobs1, alpha=alpha, ratio=ratio, alternative='two-sided')
    # Calculate MDE detectable with 80% power and given sample size
    mde = zt_ind_solve_power(effect_size=None, power=0.8, nobs1=nobs1, alpha=alpha, ratio=ratio, alternative='two-sided')

    return {
        metric_col_name: {
            'prop': f'[c_{prop1:.3f}, t_{prop2:.3f}]',
            'rel_impr': f'{(prop2 - prop1) / prop1:.1%}',
            'abs_impr': f'{prop2 - prop1:.1%}',
            'pval': f'{pval:.3f}', 
            'effect': f'{effect_size:.2f} ({'small' if abs(effect_size) < 0.20 else 'medium' if abs(effect_size) < 0.50 else 'large'})',
            'ci_diff': f'[{ci_low:.3f}, {ci_upp:.3f}]', 
            'power': f'{power:.2f}',
            'mde_gap': f'{abs(np.clip(effect_size / mde - 1, a_min=None, a_max=0)):.1%}',
            'visual': binary_plot(variant_metric, metric_col_name)
        }
    }

def chi_squared_test(variant_metric: pd.DataFrame, categ_heatmap_plot_catvarname: str=None) -> dict:
    """
    Chi-squared test for categorical variables with an Effect Size (Cramer's V, Cohen's w) and Power Analysis.  
    Parameter "categ_heatmap_plot_catvarname": specifies the original name of the categorical variable used in the heatmap plot.
    """
    
    catvar_col_name = variant_metric.iloc[:, 1].name

    # Create a contingency table
    contingency = pd.crosstab(variant_metric['variant'], variant_metric[catvar_col_name])

    # Perform the chi-squared test
    chi2, pval, dof, expected = chi2_contingency(contingency, correction=False) # correction=False for large sample sizes to avoid Yates' correction for continuity, which is conservative and reduces power
    
    # Calculate effect size (Cramer's V)
    # Cramer's V is a conservative measure of association between two categorical variables
        # V ≈ 0.00-0.10 → Very small effect
        # V ≈ 0.10-0.30 → Small effect
        # V ≈ 0.30-0.50 → Medium effect
        # V > 0.50 → Large effect
    # *interpretation depends on the number of categories (k) – the more categories, the smaller the effect size for the same association strength
    n = contingency.sum().sum() # total number of observations
    cramer_v = np.sqrt(chi2 / (n * min(contingency.shape) - 1))
    
    # Power analysis for the chi-square independence test
    k = contingency.shape[1] # number of categories
    cohen_w = cramer_v * np.sqrt(k - 1) # Cramer's V to Cohen's w conversion (used when calculating power)
    alpha = 0.05
    # Calculate power for given effect size, sample, and alpha
    power = GofChisquarePower().solve_power(effect_size=cohen_w, power=None, nobs=n, alpha=alpha, n_bins=dof + 1) # required tweak: n_bins = dof + 1 for GofChisquarePower in homogeneity/independence tests
    # Calculate MDE detectable with 80% power and given sample size
    mde = GofChisquarePower().solve_power(effect_size=None, power=0.8, nobs=n, alpha=alpha, n_bins=dof + 1)

    return {
        catvar_col_name: {
            'categs': contingency.columns.tolist(),
            'prop': [f'{variant_metric[catvar_col_name].value_counts(normalize=True).loc[dt]:.1%}' for dt in contingency.columns],
            'pval': f'{pval:.3f}',
            'effect': f'{cramer_v:.2f} ({'negligible' if cramer_v < 0.10 else 'small' if cramer_v < 0.30 else 'medium' if cramer_v < 0.50 else 'large'})',
            'power': f'{power:.2f}',
            'mde_gap': f'{abs(np.clip(cohen_w / mde - 1, a_min=None, a_max=0)):.1%}',
            'visual': categ_heatmap_plot(variant_metric, catvar_col_name, categ_heatmap_plot_catvarname if categ_heatmap_plot_catvarname != None else None)
        }
    }

def brunner_munzel_test(variant_metric: pd.DataFrame) -> dict:
    """Brunner-Munzel test for normal/non-normal distributions and equal/unequal variances with an Effect Size (Pest p*/Cliff's δ)."""
    
    metric_col_name = variant_metric.iloc[:, 1].name
    
    # If treatment column is actually called differently
    treatment_col = [col for col in variant_metric.variant.unique() if col != 'control'][0]

    group1 = variant_metric.query('variant == "control"')[metric_col_name]
    group2 = variant_metric.query('variant == @treatment_col')[metric_col_name]
    
    # Performing the Brunner-Munzel test
    stat, pval = brunnermunzel(group1, group2, alternative='two-sided')
    
    # Calculating the effect size (Pest/p*) as the probability of group2 being greater than group1 (stochastic superiority)
    # Pest (i.e. probability estimation) serves as a rank-based probabilistic effect size that complements the Brunner-Munzel test
        # Ranges from 0 to 1
        # p* = 0.5 = no effect (called "stochastic equality") = the probability that Y is greater than X equals the probability that X is greater than Y.
        # p* > 0.5 → values in Y tend to be larger than X
        # p* < 0.5 → values in X tend to be larger than Y
    nx, ny = len(group1), len(group2)
    all_data = np.concatenate([group1, group2])
    ranks = stats.rankdata(all_data)  # Automatically handles ties
    rank_y = ranks[nx:]  # Ranks for group y
    mean_rank_y = np.mean(rank_y)
    effect_pest = (mean_rank_y - (ny + 1)/2) / nx # Derived from the original effect size (Pest) formula P(X<Y)+0.5⋅P(X=Y)

    # Convert probability-based effect size (Pest) to straightforward Cliff’s Delta (-1 to 1 where 0 indicates no effect)
        # δ > 0 → Y tends to dominate X (e.g. +0.5 δ = 0.75 Pest)
        # δ < 0 → X tends to dominate Y (e.g. -0.5 δ = 0.25 Pest)
        # δ ≈ 0 → No dominance (0 = 0.5 Pest = stochastic equality)
    effect_delta = 2 * effect_pest - 1 

    power, kde_plot = monte_carlo_power_analysis(variant_metric)

    return {
        metric_col_name: {
            'pval': f'{pval:.3f}',
            'effect': f'{np.clip(effect_delta, -1.0, 1.0):.2f} ({'c over t' if abs(effect_delta) < 0 else 'no effect' if abs(effect_delta) == 0 else 't over c'})',
            'power': f'{power:.2f}',
            'visual': kde_plot
        }
    }

def monte_carlo_power_analysis(variant_metric: pd.DataFrame, n_simulations=100, n_datapoints=10_000, alpha=0.05) -> float:
    """
    Performs Monte Carlo simulation for power analysis on metrics, using different methods based on data type:
    * KDE – Kernel Density Estimation with a Gaussian kernel (for discretized (clustered) continuous metrics).
    * SRS – Simple Random Sampling with replacement (for other numerical metrics; classical approach for large samples).\n
    Each method estimates the theoretical distribution of the data **to assess statistical power**.\n
    Brunner-Munzel test is used for group comparison.  
    Test has more power in the KDE case as continuous values substantially reduce the number of ties (repeated integers).
    """
    
    metric_col_name = variant_metric.iloc[:, 1].name
    
    # If treatment column is actually called differently
    treatment_col = [col for col in variant_metric.variant.unique() if col != 'control'][0]

    # Extracting the control and treatment groups
    group1 = variant_metric.query('variant == "control"')[metric_col_name]
    group2 = variant_metric.query('variant == @treatment_col')[metric_col_name]
    
    def power_calculation(control, treatment, n_simulations=n_simulations, n_datapoints=n_datapoints, alpha=alpha, estimator='SRS'):
        """Power calculation based on the selected estimation method (KDE/SRS)"""
        p_values = []

        if estimator == 'KDE':
            # Sampling based on estimations and Cutting out generated values below 0 (Gaussian KDE extends to ±∞)
            sampled_kde = lambda kde, n: np.clip(kde.resample(n).flatten(), a_min=0, a_max=None)
            for _ in range(n_simulations):
                sample_control = sampled_kde(control, n_datapoints)
                sample_treatment = sampled_kde(treatment, n_datapoints)

                stat, pval = brunnermunzel(sample_control, sample_treatment)
                p_values.append(pval)
        else:
            for _ in range(n_simulations):
                sample_control = control.sample(n_datapoints, replace=True)
                sample_treatment = treatment.sample(n_datapoints, replace=True)
            
                stat, pval = brunnermunzel(sample_control, sample_treatment)
                p_values.append(pval)
        
        power = np.mean(np.array(p_values) < alpha)
        return power
    
    # Creating a list of continuous traces (e.g. time, money) to check if the metric is continuous by nature
    continuous_traces = [
        # Time-related
        'time', 'dur', 'sec', 'min', 'hour', 'day', 'week', 'month', 'year', 'length', 'latency', 
        'load', 'wait', 'uptime', 'downtime', 'interval', 'elapsed', 'runtime', 'delay', 'period',
        # Money-related
        'price', 'cost', 'rev', 'profit', 'spend', 'spent', 'amount', 'pay', 'income', 
        'earn', 'margin', 'roi', 'charge', 'fee', 'salary', 'discount'
    ]

    # Checking if the metric is continuous by the presence of continuous traces in its name (which is an indicator of a discretized continuity)
    # Continuous metrics will be omitted to avoid smoothing the already smooth distribution
    if variant_metric[metric_col_name].dtype.kind != 'f' and any(trace in metric_col_name.lower() for trace in continuous_traces):
        # KDE is a way to estimate the probability distribution of a dataset by placing smooth,...
        # ...bell-shaped curves (Gaussians) on each data point and adding them up to get a smooth continuous overall curve.

        # KDE’s default bandwidth (smoothing factor) undersmooths discretized continuous values (e.g., clustered rounded minutes). Adjusting it better approximates the true continuous distribution.
        adjusted_bw_factor = lambda data: 1.5 * gaussian_kde(data, bw_method='scott').scotts_factor() # 1.5 factor is an heuristic for cluster mild oversmoothing

        # Adjusted estimations
        kde_control = gaussian_kde(group1, bw_method=adjusted_bw_factor(group1))
        kde_treatment = gaussian_kde(group2, bw_method=adjusted_bw_factor(group2))

        # Calculate power
        return power_calculation(kde_control, kde_treatment, estimator='KDE'), kde_plot(variant_metric, metric_col_name, mcarlo_kde=True)

    else:
        # Simple Random Sampling (SRS) with replacement: original data preserved, no discretized continuous values. Optimal since no artificial value clustering. 

        # Calculate power
        return power_calculation(group1, group2, estimator='SRS'), kde_plot(variant_metric, metric_col_name, mcarlo_kde=False)

def two_part_testing(variant_metric: pd.DataFrame) -> dict:
    """Two-part testing for zero-inflated distributions with a Chi-squared test for zeros and Brunner-Munzel test for non-zero values."""
      
    metric_col_name = variant_metric.iloc[:, 1].name

    # The first part of the two-part testing ––––––––––––––––––––––––––––––––––––
    # Counting the number of zeros and non-zeros in the metric
    variant_metric = variant_metric.copy() # to avoid SettingWithCopyWarning
    variant_metric.loc[:, 'zero_nonzero'] = np.where(variant_metric[metric_col_name] == 0, 'Zero value', 'Non-zero value')
    
    # Performing the chi-squared test for proportions of zeros
    chi_squared_result = chi_squared_test(variant_metric[['variant', 'zero_nonzero']], categ_heatmap_plot_catvarname=metric_col_name)

    # Renaming the key for clarity and further ease of processing for reporting
    chi_squared_result['zeros'] = chi_squared_result.pop('zero_nonzero')

    # The second part of the two-part testing ––––––––––––––––––––––––––––––––––––
    # Performing the Brunner-Munzel test for non-zero values
    brunner_munzel_result = brunner_munzel_test(variant_metric[variant_metric[metric_col_name] != 0][['variant', metric_col_name]])

    # Renaming the key for clarity and further ease of processing for reportings
    brunner_munzel_result['non-zeros'] = brunner_munzel_result.pop(metric_col_name)

    return {
        metric_col_name: {
            **chi_squared_result,
            **brunner_munzel_result
        }
    }

In [246]:
# Visualization functions (called within the stat. method functions and serve for Excel-based reporting)

def binary_plot(variant_metric: pd.DataFrame, metric_col_name: str) -> plt.Figure:
    """Plots group-wise binary proportions with 95% CI."""
    
    # Get treatment column
    variants = variant_metric['variant'].unique()
    treatment_col = variants[variants != 'control'][0]
    
    # Sort data
    sort_order = variant_metric['variant'].replace({'control': 0, treatment_col: 1})
    df = variant_metric.iloc[sort_order.argsort()].copy()
    
    # Calculate statistics and CIs in one pass
    group_stats = df.groupby('variant', sort=False)[metric_col_name].agg(['size', 'mean'])
    group_stats['ci'] = 1.96 * np.sqrt(group_stats['mean'] * (1 - group_stats['mean']) / group_stats['size'])
    
    # Create plot
    fig, ax = plt.subplots(figsize=(5, 5))
    x_pos = np.arange(len(group_stats))
    colors = plt.cm.tab10.colors[:len(group_stats)]

    # Specifying the metric name
    ax.text(x=-0.1, y=1.1, s=metric_col_name, alpha=0.3, transform=ax.transAxes)
    
    # Plot with pre-calculated CIs
    for i, (group, stats) in enumerate(group_stats.iterrows()):
        ax.errorbar(x_pos[i], stats['mean'], yerr=stats['ci'],
                   fmt='o', color=colors[i], markersize=8, elinewidth=2)
    
    # Formatting
    ax.set_ylim(0, 1)
    ax.set_xlim(x_pos[0] - 0.5, x_pos[-1] + 0.5)
    ax.set_title('Proportion by test group with 95% CI')
    ax.set_ylabel('Proportion (success rate)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(group_stats.index)
    ax.spines[['top','right','bottom','left']].set_visible(False)
    ax.tick_params(length=0, pad=5)
    ax.grid(visible=True, axis='x', alpha=0.15, linewidth=2)
    
    # Annotations and CI boundary line
    upper_bounds = group_stats['mean'] + group_stats['ci']
    min_upper = upper_bounds.min()
    
    for i, (group, row) in enumerate(group_stats.iterrows()):
        ax.text(x_pos[i] - 0.1, row['mean'], f"{row['mean']:.2f}",
                ha='right', va='center', fontsize=10, alpha=0.65)
    
    ax.axhline(min_upper, color='grey', linestyle='--', 
               linewidth=1.5, alpha=0.35, zorder=1)
    
    # Legend
    format_number = lambda n: f'{n/1e6:.1f}M' if n >= 1e6 else f'{n/1e3:.1f}K' if n >= 1e3 else str(n)
    legend_elements = [
        Line2D([0], [0], marker='o', color=colors[i], 
               label=f'{group} (n={format_number(row["size"])})',
               markersize=8, linestyle='None') 
        for i, (group, row) in enumerate(group_stats.iterrows())
    ]
    legend_elements.append(
        Line2D([0], [0], linestyle='--', color='grey',
               label='min upper CI boundary', alpha=0.6)
    )
    ax.legend(handles=legend_elements, frameon=True, bbox_to_anchor=(1.35, 0.95))
    
    plt.close(fig)

    return fig

def categ_heatmap_plot(variant_metric: pd.DataFrame, metric_col_name: str, categ_heatmap_plot_catvarname: str=None) -> plt.Figure:
    """Displays a row-wise proportion heatmap with test group differences and total counts for categories."""
    
    df = pd.crosstab(variant_metric[metric_col_name], variant_metric['variant'])
    row_sums = df.sum(axis=1)
    df_prop = df.div(row_sums, axis=0)

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.7, 1.8*len(df.index)), gridspec_kw={'width_ratios':[3,1.5], 'wspace':0.05}, sharey=True)

    # HEATMAP (Left)
    sns.heatmap(df_prop, annot=True, fmt=".2f", cmap="Blues", linewidths=1.5, cbar=False, ax=ax1, annot_kws={'size': 10})

    # Formatting
    ax1.set(yticklabels=df.index, xlabel='', ylabel='', title='Proportion heatmap (rows)')
    ax1.tick_params(length=0, pad=5)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

    # Specifying the metric name
    ax1.text(x=-0.1, y=1.1, s=categ_heatmap_plot_catvarname if categ_heatmap_plot_catvarname != None else metric_col_name, alpha=0.3, transform=ax1.transAxes)

    # Calculate absolute differences between control and treatment groups
    control_col = 'control'
    treatment_col = next(col for col in df.columns if col != 'control')

    # Add colored circles with differences
    for i, (idx, row) in enumerate(df_prop.iterrows()):
        if control_col in df_prop.columns:
            diff = row[treatment_col] - row[control_col]
            abs_diff = abs(diff)
            
            # Position between control and treatment columns
            control_idx = list(df_prop.columns).index(control_col)
            treatment_idx = list(df_prop.columns).index(treatment_col)
            x_pos = (control_idx + treatment_idx) / 2
            
            # Determine circle color (green if treatment improved)
            circle_color = "#548553" if round(diff, 2) > 0 else "#7A7A7A" if round(diff, 2) == 0 else "#9F4D4E" # green/grey/red
            
            # Add circle with difference value
            circle = plt.Circle((1.02, i+0.5), 0.2, color=circle_color, alpha=1)
            ax1.add_patch(circle)
            
            # Add text inside circle
            ax1.text(1.02, i+0.5, f'{abs_diff:.2f}', 
                    ha='center', va='center', color='white', fontsize=9, fontweight='bold')

    # BAR PLOT (Right)
    bar_height = 0.6
    y_offset = (1 - bar_height)/2
    y_pos = np.arange(len(df)) + y_offset

    bars = ax2.barh(y_pos, row_sums, height=bar_height, align='edge',
                    color="#3a6ea5", alpha=0.9, edgecolor='white')

    # Formatting
    ax2.set(title='Total count', xlabel='', xticks=[])
    ax2.yaxis.set_visible(False)
    ax2.spines[['top','right','bottom']].set_visible(False)
    ax2.spines['left'].set(color='#607fa5', alpha=0.3, linewidth=2)

    # Cleaner number formatting
    format_number = lambda value: f'{value / 1_000_000:.1f}M' if abs(value) >= 1_000_000 else f'{value / 1_000:.1f}K'

    max_width = row_sums.max()
    text_offset = max_width * 0.04  # Dynamic spacing

    for bar, val in zip(bars, row_sums):
        x_pos = bar.get_width() + text_offset
        ax2.text(x_pos, bar.get_y() + bar.get_height()/2, format_number(val), 
                va='center', ha='left', color="#3B587B", size=10, weight='bold', alpha=0.5)

    ax2.set_xlim([0, max_width * 1.2])  # Auto-adjust xlim
    plt.close(fig)

    return fig

def kde_plot(variant_metric: pd.DataFrame, metric_col_name: str, mcarlo_kde: bool=False) -> plt.Figure:
    """
    Create KDE plots comparing control vs treatment probability distributions.\n
    Paramater "mcarlo_kde": Whether to show the smoothed version of a discretized continuous metric with clusters (second subplot) after Monte Carlo power analysis.
    """

    # Get treatment column
    variants = variant_metric['variant'].unique()
    treatment_col = variants[variants != 'control'][0]

    # Extracting the control and treatment groups
    group1 = variant_metric.query('variant == "control"')[metric_col_name]
    group2 = variant_metric.query('variant == @treatment_col')[metric_col_name]
    group_sizes = variant_metric['variant'].value_counts().to_dict()

    fig, axes = plt.subplots(nrows=1, ncols=2 if mcarlo_kde else 1, figsize=(5.5 + 5.5*mcarlo_kde, 5), sharey=True, gridspec_kw={'wspace': 0.05})
    axes = np.array(axes).flatten() # ensure axes is always an array, otherwise "TypeError: 'Axes' object is not subscriptable" might occur

    # Discrete/continuous values probability distribution
    sns.kdeplot(group1, ax=axes[0])
    sns.kdeplot(group2, ax=axes[0])

    # Formatting
    axes[0].set(ylabel='Density (occurrence probability)', title='Probability distribution\n(before smoothed clusters)' if mcarlo_kde else 'Probability distribution')

    # Specifying the metric name
    axes[0].text(x=-0.1, y=1.15 if mcarlo_kde else 1.1, s=metric_col_name, alpha=0.3, transform=axes[0].transAxes)

    # Legend building
    format_number = lambda value: f'{value / 1_000_000:.1f}M' if abs(value) >= 1_000_000 else f'{value / 1_000:.1f}K'
    legend_labels = [
        f'control (n={format_number(group_sizes["control"])})',
        f'{treatment_col} (n={format_number(group_sizes[treatment_col])})'
    ]
    axes[0].legend(labels=legend_labels, bbox_to_anchor=(1.05, 0.98) if mcarlo_kde else (1, 0.98))

    if mcarlo_kde:
        # Discretized continuous values probability distribution (with discrete clusters smoothed by adjusted KDE smoothing factor)
        sns.kdeplot(group1, bw_adjust=1.5, ax=axes[1])
        sns.kdeplot(group2, bw_adjust=1.5, ax=axes[1])

        # Formatting
        axes[1].set(ylabel='Density (occurrence probability)', title='Probability distribution\n(after smoothed clusters)')
        axes[1].text(0.2, 0.87, s='smoothing discretized continuous values\nfor better population approximation during\npower analysis in a Monte Carlo simulation', 
                     alpha=0.3, transform=axes[1].transAxes)

    # Common formatting for all axes
    for ax in axes:
        ax.spines[['top','right','bottom','left']].set_visible(False)
        ax.tick_params(length=0, pad=5)

    plt.close(fig)

    return fig
        

In [247]:
# Extracting primary and secondary metrics
target_metrics_list = ab_test.columns[ab_test.columns.str.match(r'^(sm|pm)')].to_list()

dtype_dstat_dict = {}
# Calculating the type of statistical test for each metric based on its data type
for metric in target_metrics_list:
    variant_metric = ab_test[['variant', metric]]

    # Checking if the metric is binary
    if set(variant_metric[metric].dropna().unique()) <= set([0, 1]):
        if 'binary' in dtype_dstat_dict:
            dtype_dstat_dict['binary'].append(z_test_prop(variant_metric))
        else:
            dtype_dstat_dict['binary'] = [z_test_prop(variant_metric)]

    # Checking if the metric is object/category
    elif variant_metric[metric].dtype.kind in {'O', 'c'}:
        if 'categorical' in dtype_dstat_dict:
            dtype_dstat_dict['categorical'].append(chi_squared_test(variant_metric))
        else:
            dtype_dstat_dict['categorical'] = [chi_squared_test(variant_metric)]

    # Checking if the metric is integer/float
    elif variant_metric[metric].dtype.kind in {'i', 'f'}:
        # Pulling out value proportions to check if the metric data is zero-inflated
        value_prop = variant_metric.value_counts(normalize=True).unstack()

        # Checking if the metric is zero-inflated (group-wise)
        # Moderately-to-highly zero-inflated distribution (>20% of zeros)
        if 0 in value_prop.columns and all(value_prop.loc[:, 0] > 0.2):
            if 'zero_inflated' in dtype_dstat_dict:
                # Chi-squared test for proportions of zeros and Brunner-Munzel test for non-zero values, hence "Two-Part Testing"
                dtype_dstat_dict['zero_inflated'].append(two_part_testing(variant_metric))
            else:
                dtype_dstat_dict['zero_inflated'] = [two_part_testing(variant_metric)]
        # Non-to-low zero-inflated distribution
        else:
            if 'non_to_moderately_zero_inflated' in dtype_dstat_dict:
                dtype_dstat_dict['non_to_low_zero_inflated'].append(brunner_munzel_test(variant_metric))
            else:
                dtype_dstat_dict['non_to_low_zero_inflated'] = [brunner_munzel_test(variant_metric)]

    else:
        raise ValueError(f"Unsupported data type for metric '{metric}': type({variant_metric[metric].dtype}), values({variant_metric[metric].unique()[:5]}).")

In [248]:
# Test metrics descriptions
test_metrics_description = {
    'binary': {
        'test_used': 'Z-test for proportions',
        'test_purpose': 'Tests difference in proportions',
        'effect_used': 'Cohen\'s h',
        'prop': 'Successes proportion of control and treatment groups',
        'rel_impr': 'Relative improvement',
        'abs_impr': 'Absolute improvement',
        'pval': 'Test p-value',
        'effect': 'Cohen\'s h effect size (magnitude of proportions difference)',
        'ci_diff': '95% CI for the proportions difference',
        'power': 'Test power for the two-sample z-test',
        'mde_gap': 'Gap between MDE and observed effect size'
    },
    'categorical': {
        'test_used': 'Chi-squared test',
        'test_purpose': 'Tests association between categories',
        'effect_used': 'Cramer\'s V, Cohen\'s w (for power analysis)',
        'categs': 'Categories being compared',
        'prop': 'Total proportion of categories respectively',
        'pval': 'Test p-value',
        'effect': 'Cramer\'s V effect size (association strength)',
        'power': 'Test power for the two-sample chi-squared test',
        'mde_gap': 'Gap between MDE and observed effect size'
    },
    'zero_inflated': {
        'zeros': {
            'test_used': 'Chi-squared test',
            'test_purpose': 'Tests association between categories',
            'effect_used': 'Cramer\'s V, Cohen\'s w (for power analysis)',
            'categs': 'Categories being compared',
            'prop': 'Total proportion of categories respectively',
            'pval': 'Test p-value',
            'effect': 'Cramer\'s V effect size (association strength)',
            'power': 'Test power for the two-sample chi-squared test',
            'mde_gap': 'Gap between MDE and observed effect size (both Cohen\'s w)'
        },
        'non-zeros': {
            'test_used': 'Brunner-Munzel test',
            'test_purpose': 'Tests dominance between groups',
            'effect_used': 'Cliff\'s Delta',
            'pval': 'Test p-value',
            'effect': 'Cliff\'s Delta effect size (magnitude of group dominance)',
            'power': 'Monte Carlo-based test power estimate'
        }
    },
    'non_to_low_zero_inflated': {
        'test_used': 'Brunner-Munzel test',
        'test_purpose': 'Tests dominance between groups',
        'effect_used': 'Cliff\'s Delta',
        'pval': 'Test p-value',
        'effect': 'Cliff\'s Delta effect size (magnitude of group dominance)',
        'power': 'Monte Carlo-based test power estimate'
    }
}

def matplotlib_to_excel_image(fig, sheet, start_row, start_col, has_subplots=False):
    """
    Converts matplotlib figure to Excel image and inserts it at specified position
    """
    try:
        # Save figure to memory buffer
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        
        # Create openpyxl Image directly from buffer
        img = Image(buf)
        
        # Calculate position
        col_letter = chr(64 + start_col)  # Convert column number to letter
        cell_ref = f"{col_letter}{start_row}"
        
        # Preserve original proportions but reduce size by 30% for single plots
        original_width = img.width
        original_height = img.height
        
        if not has_subplots:
            # Reduce by 30% for single subplot figures
            img.width = int(original_width * 0.7)
            img.height = int(original_height * 0.7)
        else:
            # Keep original size for multi-subplot figures
            # Only resize if too large
            if img.width > 600:
                aspect_ratio = img.height / img.width
                img.width = 600
                img.height = int(600 * aspect_ratio)
            
            if img.height > 500:
                aspect_ratio = img.width / img.height
                img.height = 500
                img.width = int(500 * aspect_ratio)
        
        # Add dark grey border
        from openpyxl.styles import Side
        img.border = Side(style='thick', color='404040')  # Dark grey border
        
        # Add image to sheet
        sheet.add_image(img, cell_ref)
        
        # Calculate how many rows the image spans based on actual height
        rows_spanned = max(12, int(img.height / 18))
        
        return rows_spanned
        
    except Exception as e:
        print(f"Error adding image: {e}")
        return 12  # Default spacing if image fails

def create_description_table(ws, data_type, current_row, test_descriptions):
    """
    Creates description table for test metrics
    """
    description_font = Font(bold=True, size=10, color='333333')
    description_fill = PatternFill(start_color='E6F3FF', end_color='E6F3FF', fill_type='solid')
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    if data_type == 'zero_inflated':
        # Handle zero_inflated with both zeros and non-zeros horizontally
        # Headers
        ws.cell(row=current_row, column=1, value="Statistic")
        ws.cell(row=current_row, column=2, value="Zeros Description")
        ws.cell(row=current_row, column=3, value="Non-zeros Description")
        
        for col in range(1, 4):
            cell = ws.cell(row=current_row, column=col)
            cell.font = description_font
            cell.fill = description_fill
            cell.border = border
            cell.alignment = Alignment(horizontal='center', vertical='center')
        
        current_row += 1
        
        # Get all unique statistics from both zeros and non-zeros
        zeros_desc = test_descriptions['zeros']
        non_zeros_desc = test_descriptions['non-zeros']
        all_stats = set(list(zeros_desc.keys()) + list(non_zeros_desc.keys()))
        
        for stat in all_stats:
            ws.cell(row=current_row, column=1, value=stat)
            ws.cell(row=current_row, column=2, value=zeros_desc.get(stat, ''))
            ws.cell(row=current_row, column=3, value=non_zeros_desc.get(stat, ''))
            
            for col in range(1, 4):
                cell = ws.cell(row=current_row, column=col)
                cell.border = border
                cell.font = Font(size=9)
                if current_row % 2 == 0:
                    cell.fill = PatternFill(start_color='F8F8F8', end_color='F8F8F8', fill_type='solid')
            
            current_row += 1
    else:
        # Regular description table
        ws.cell(row=current_row, column=1, value="Statistic")
        ws.cell(row=current_row, column=2, value="Description")
        
        for col in range(1, 3):
            cell = ws.cell(row=current_row, column=col)
            cell.font = description_font
            cell.fill = description_fill
            cell.border = border
            cell.alignment = Alignment(horizontal='center', vertical='center')
        
        current_row += 1
        
        # Add metadata rows first (test_used, test_purpose, effect_used)
        metadata_items = ['test_used', 'test_purpose', 'effect_used']
        for meta_key in metadata_items:
            if meta_key in test_descriptions:
                ws.cell(row=current_row, column=1, value=meta_key)
                ws.cell(row=current_row, column=2, value=test_descriptions[meta_key])
                
                for col in range(1, 3):
                    cell = ws.cell(row=current_row, column=col)
                    cell.border = border
                    cell.font = Font(size=9)
                    if current_row % 2 == 0:
                        cell.fill = PatternFill(start_color='F8F8F8', end_color='F8F8F8', fill_type='solid')
                
                current_row += 1
        
        # Then add other statistics
        for stat, description in test_descriptions.items():
            if stat not in metadata_items:  # Skip metadata that we already added
                ws.cell(row=current_row, column=1, value=stat)
                ws.cell(row=current_row, column=2, value=description)
                
                for col in range(1, 3):
                    cell = ws.cell(row=current_row, column=col)
                    cell.border = border
                    cell.font = Font(size=9)
                    if current_row % 2 == 0:
                        cell.fill = PatternFill(start_color='F8F8F8', end_color='F8F8F8', fill_type='solid')
                
                current_row += 1
    
    return current_row + 1

def create_excel_report(data_dict, ab_test_df=None, output_filename='ab_test_results_repot.xlsx'):
    """
    Disassembles a nested dictionary containing A/B test results and exports to Excel  
    with proper formatting and separate sections for each data type.
    """
    
    # Create a new workbook
    wb = Workbook()
    
    # Remove default sheet and create our sheets
    wb.remove(wb.active)
    summary_ws = wb.create_sheet("Summary")
    detailed_ws = wb.create_sheet("Detailed Results")
    
    # Set Summary sheet tab color to gentle green-grey
    summary_ws.sheet_properties.tabColor = "A8C090"
    
    current_row = 1
    
    # Add sample size summary table if ab_test_df is provided
    if ab_test_df is not None:
        # Create variant summary
        variant_counts = ab_test_df['variant'].value_counts()
        total_samples = len(ab_test_df)
        
        # Sample Size Summary Table Header
        summary_ws.cell(row=current_row, column=1, value="Sample Size Summary")
        summary_ws.cell(row=current_row, column=1).font = Font(bold=True, size=14)
        summary_ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
        summary_ws.merge_cells(f'A{current_row}:C{current_row}')
        current_row += 2
        
        # Table headers
        headers = ['Variant', 'Sample Size', 'Ratio']
        for col_idx, header in enumerate(headers, 1):
            cell = summary_ws.cell(row=current_row, column=col_idx, value=header)
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
            cell.border = Border(
                left=Side(style='thin'),
                right=Side(style='thin'),
                top=Side(style='thin'),
                bottom=Side(style='thin')
            )
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # Add variant data
        for variant in ['control', 'treatment']:
            if variant in variant_counts:
                sample_size = variant_counts[variant]
                ratio = f"{(sample_size / total_samples * 100):.1f}%"
                
                # Variant name (use full names instead of c/t)
                cell = summary_ws.cell(row=current_row, column=1, value=variant)
                cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
                cell.alignment = Alignment(horizontal='center', vertical='center')
                
                # Sample size
                cell = summary_ws.cell(row=current_row, column=2, value=sample_size)
                cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
                cell.alignment = Alignment(horizontal='center', vertical='center')
                
                # Ratio
                cell = summary_ws.cell(row=current_row, column=3, value=ratio)
                cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
                cell.alignment = Alignment(horizontal='center', vertical='center')
                
                # Alternate row colors
                if current_row % 2 == 0:
                    for col in range(1, 4):
                        summary_ws.cell(row=current_row, column=col).fill = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
                
                current_row += 1
        
        current_row += 3  # Add space before main results
    
    # Main Results Header
    summary_ws.cell(row=current_row, column=1, value="AB Test Results Summary")
    summary_ws.cell(row=current_row, column=1).font = Font(bold=True, size=14)
    summary_ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
    
    # Merge with adjacent right cell to widen the title
    summary_ws.merge_cells(f'A{current_row}:B{current_row}')
    current_row += 2
    
    # Create summary table
    summary_data = []
    
    for data_type, metrics_list in data_dict.items():
        for metric_dict in metrics_list:
            for metric_name, stats_data in metric_dict.items():
                if data_type == 'zero_inflated' and isinstance(stats_data, dict) and ('zeros' in stats_data or 'non-zeros' in stats_data):
                    # Handle zero_inflated metrics with nested dictionaries
                    if 'zeros' in stats_data:
                        row_data = {'Data Type': data_type.capitalize(), 'Metric': f"{metric_name} (zeros)"}
                        zeros_stats = stats_data['zeros']
                        for stat_name, stat_value in zeros_stats.items():
                            if stat_name != 'visual':
                                # Convert lists to strings for Excel compatibility
                                if isinstance(stat_value, list):
                                    row_data[stat_name] = ', '.join(map(str, stat_value))
                                else:
                                    row_data[stat_name] = stat_value
                        summary_data.append(row_data)
                    
                    if 'non-zeros' in stats_data:
                        row_data = {'Data Type': data_type.capitalize(), 'Metric': f"{metric_name} (non-zeros)"}
                        non_zeros_stats = stats_data['non-zeros']
                        for stat_name, stat_value in non_zeros_stats.items():
                            if stat_name != 'visual':
                                # Convert lists to strings for Excel compatibility
                                if isinstance(stat_value, list):
                                    row_data[stat_name] = ', '.join(map(str, stat_value))
                                else:
                                    row_data[stat_name] = stat_value
                        summary_data.append(row_data)
                else:
                    # Regular metrics
                    row_data = {'Data Type': data_type.capitalize(), 'Metric': metric_name}
                    for stat_name, stat_value in stats_data.items():
                        if stat_name != 'visual':
                            # Convert lists to strings for Excel compatibility
                            if isinstance(stat_value, list):
                                row_data[stat_name] = ', '.join(map(str, stat_value))
                            else:
                                row_data[stat_name] = stat_value
                    summary_data.append(row_data)
    
    # Create DataFrame and write to summary sheet
    if summary_data:
        df_summary = pd.DataFrame(summary_data)
        
        # Determine number of columns for header merging
        max_cols = len(df_summary.columns)
        summary_ws.merge_cells(f'A{current_row-1}:{chr(64 + max_cols)}{current_row-1}')
        
        # Write headers
        headers = list(df_summary.columns)
        for col_idx, header in enumerate(headers, 1):
            cell = summary_ws.cell(row=current_row, column=col_idx, value=header)
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
            cell.border = Border(
                left=Side(style='thin'),
                right=Side(style='thin'),
                top=Side(style='thin'),
                bottom=Side(style='thin')
            )
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # Write data
        for row_idx, row in enumerate(df_summary.itertuples(index=False), current_row):
            for col_idx, value in enumerate(row, 1):
                cell = summary_ws.cell(row=row_idx, column=col_idx, value=value)
                cell.border = Border(
                    left=Side(style='thin'),
                    right=Side(style='thin'),
                    top=Side(style='thin'),
                    bottom=Side(style='thin')
                )
                if row_idx % 2 == 0:
                    cell.fill = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
        
        # Auto-adjust column widths
        for col_num in range(1, len(headers) + 1):
            max_length = 0
            column_letter = chr(64 + col_num)  # Convert to letter (A, B, C, etc.)
            
            # Check header length
            if col_num <= len(headers):
                max_length = max(max_length, len(str(headers[col_num - 1])))
            
            # Check data lengths
            for row_idx in range(current_row, current_row + len(df_summary)):
                try:
                    cell = summary_ws.cell(row=row_idx, column=col_num)
                    if cell.value is not None:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            
            adjusted_width = min(max_length + 2, 30)
            summary_ws.column_dimensions[column_letter].width = adjusted_width
    
    # Now create the detailed sheet
    current_row = 1
    
    # Define styles
    header_font = Font(bold=True, size=12, color='FFFFFF')
    header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
    
    metric_font = Font(bold=True, size=11, color='FFFFFF')
    metric_fill = PatternFill(start_color='4F81BD', end_color='4F81BD', fill_type='solid')
    
    stat_font = Font(size=10)
    stat_fill = PatternFill(start_color='DCE6F1', end_color='DCE6F1', fill_type='solid')
    
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    center_alignment = Alignment(horizontal='center', vertical='center')
    
    # Process each data type for detailed sheet
    for data_type, metrics_list in data_dict.items():
        # Add section header (without test names)
        section_header = f"{data_type.upper()} VARIABLES"
        detailed_ws.cell(row=current_row, column=1, value=section_header)
        detailed_ws.cell(row=current_row, column=1).font = header_font
        detailed_ws.cell(row=current_row, column=1).fill = header_fill
        detailed_ws.cell(row=current_row, column=1).border = border
        detailed_ws.cell(row=current_row, column=1).alignment = center_alignment
        
        # Merge cells for section header
        detailed_ws.merge_cells(f'A{current_row}:C{current_row}')
        current_row += 2
        
        # Add description table
        if data_type in test_metrics_description:
            current_row = create_description_table(detailed_ws, data_type, current_row, test_metrics_description[data_type])
        
        # Process each metric
        for metric_dict in metrics_list:
            for metric_name, stats_data in metric_dict.items():
                
                if data_type == 'zero_inflated' and isinstance(stats_data, dict) and ('zeros' in stats_data or 'non-zeros' in stats_data):
                    # Handle zero_inflated metrics with nested dictionaries
                    
                    # Add metric name as table header
                    detailed_ws.cell(row=current_row, column=1, value="Metric")
                    detailed_ws.cell(row=current_row, column=2, value=f"{metric_name} (zeros)")
                    detailed_ws.cell(row=current_row, column=3, value=f"{metric_name} (non-zeros)")
                    
                    # Style metric header
                    for col in range(1, 4):
                        cell = detailed_ws.cell(row=current_row, column=col)
                        cell.font = metric_font
                        cell.fill = metric_fill
                        cell.border = border
                        cell.alignment = center_alignment
                    
                    current_row += 1
                    
                    # Get all unique statistics from both zeros and non-zeros
                    zeros_stats = stats_data.get('zeros', {})
                    non_zeros_stats = stats_data.get('non-zeros', {})
                    all_stats = set(list(zeros_stats.keys()) + list(non_zeros_stats.keys()))
                    all_stats.discard('visual')  # Remove visual key
                    
                    # Add statistics horizontally
                    table_end_row = current_row
                    for stat_name in all_stats:
                        detailed_ws.cell(row=current_row, column=1, value=stat_name)
                        
                        zeros_value = zeros_stats.get(stat_name, '')
                        if isinstance(zeros_value, list):
                            zeros_value = ', '.join(map(str, zeros_value))
                        detailed_ws.cell(row=current_row, column=2, value=zeros_value)
                        
                        non_zeros_value = non_zeros_stats.get(stat_name, '')
                        if isinstance(non_zeros_value, list):
                            non_zeros_value = ', '.join(map(str, non_zeros_value))
                        detailed_ws.cell(row=current_row, column=3, value=non_zeros_value)
                        
                        # Style statistics rows
                        for col in range(1, 4):
                            cell = detailed_ws.cell(row=current_row, column=col)
                            if col == 1:
                                cell.font = stat_font
                                cell.fill = stat_fill
                                cell.alignment = center_alignment
                            else:
                                cell.font = stat_font
                            cell.border = border
                        
                        current_row += 1
                        table_end_row = current_row
                    
                    # Add one-cell gap and then visualizations below the table
                    current_row = table_end_row + 1
                    
                    # Check if figures have multiple subplots
                    zeros_has_subplots = False
                    non_zeros_has_subplots = False
                    
                    if 'visual' in zeros_stats:
                        zeros_has_subplots = len(zeros_stats['visual'].axes) > 1
                    if 'visual' in non_zeros_stats:
                        non_zeros_has_subplots = len(non_zeros_stats['visual'].axes) > 1
                    
                    max_image_rows = 0
                    if 'visual' in zeros_stats and 'visual' in non_zeros_stats:
                        # Both figures exist - place side by side with spacing
                        rows_spanned1 = matplotlib_to_excel_image(zeros_stats['visual'], detailed_ws, current_row, 1, zeros_has_subplots)
                        rows_spanned2 = matplotlib_to_excel_image(non_zeros_stats['visual'], detailed_ws, current_row, 4, non_zeros_has_subplots)  # 3 columns spacing
                        max_image_rows = max(rows_spanned1, rows_spanned2)
                    elif 'visual' in zeros_stats:
                        # Only zeros figure
                        max_image_rows = matplotlib_to_excel_image(zeros_stats['visual'], detailed_ws, current_row, 1, zeros_has_subplots)
                    elif 'visual' in non_zeros_stats:
                        # Only non-zeros figure
                        max_image_rows = matplotlib_to_excel_image(non_zeros_stats['visual'], detailed_ws, current_row, 1, non_zeros_has_subplots)
                    
                    # Update current row to account for image height
                    current_row += max_image_rows + 1
                    
                else:
                    # Regular metrics
                    # Add metric name as table header
                    detailed_ws.cell(row=current_row, column=1, value="Metric")
                    detailed_ws.cell(row=current_row, column=2, value=metric_name)
                    
                    # Style metric header
                    detailed_ws.cell(row=current_row, column=1).font = metric_font
                    detailed_ws.cell(row=current_row, column=1).fill = metric_fill
                    detailed_ws.cell(row=current_row, column=1).border = border
                    detailed_ws.cell(row=current_row, column=1).alignment = center_alignment
                    
                    detailed_ws.cell(row=current_row, column=2).font = metric_font
                    detailed_ws.cell(row=current_row, column=2).fill = metric_fill
                    detailed_ws.cell(row=current_row, column=2).border = border
                    detailed_ws.cell(row=current_row, column=2).alignment = center_alignment
                    
                    metric_start_row = current_row
                    current_row += 1
                    
                    # Track table end row
                    table_end_row = current_row
                    
                    # Add statistics
                    for stat_name, stat_value in stats_data.items():
                        if stat_name != 'visual':  # Skip visual key
                            detailed_ws.cell(row=current_row, column=1, value=stat_name)
                            
                            # Convert lists to strings for Excel compatibility
                            if isinstance(stat_value, list):
                                stat_value = ', '.join(map(str, stat_value))
                            detailed_ws.cell(row=current_row, column=2, value=stat_value)
                            
                            # Style statistics rows
                            detailed_ws.cell(row=current_row, column=1).font = stat_font
                            detailed_ws.cell(row=current_row, column=1).fill = stat_fill
                            detailed_ws.cell(row=current_row, column=1).border = border
                            detailed_ws.cell(row=current_row, column=1).alignment = center_alignment
                            
                            detailed_ws.cell(row=current_row, column=2).font = stat_font
                            detailed_ws.cell(row=current_row, column=2).border = border
                            
                            current_row += 1
                            table_end_row = current_row
                    
                    # Add one-cell gap and visualization below the table
                    if 'visual' in stats_data:
                        # Check if figure has multiple subplots
                        has_subplots = len(stats_data['visual'].axes) > 1
                        
                        image_row = table_end_row + 1  # One cell gap after table
                        rows_spanned = matplotlib_to_excel_image(stats_data['visual'], detailed_ws, image_row, 1, has_subplots)
                        current_row = image_row + rows_spanned + 1
                
                current_row += 1
        
        current_row += 2
    
    # Adjust column widths for detailed sheet
    detailed_ws.column_dimensions['A'].width = 25
    detailed_ws.column_dimensions['B'].width = 30
    detailed_ws.column_dimensions['C'].width = 30
    detailed_ws.column_dimensions['D'].width = 5  # Space for images
    detailed_ws.column_dimensions['E'].width = 5
    detailed_ws.column_dimensions['F'].width = 5
    detailed_ws.column_dimensions['G'].width = 5
    
    # Save the workbook
    full_path = os.path.abspath(output_filename)
    wb.save(output_filename)
    print(f"Excel file '{output_filename}' with summary and detailed sheets has been created!")
    print(f"Full path: {full_path}")
    
    return output_filename

# Create a report
data = dtype_dstat_dict

create_excel_report(data, ab_test_df=ab_test, output_filename='ab_test_results_repot.xlsx')

print("\nExcel file created:")
print("ab_test_results_repot.xlsx - Complete report with summary, detailed views, and visualizations")

Excel file 'ab_test_results_repot.xlsx' with summary and detailed sheets has been created!
Full path: d:\High-usage\Data Science\Data Analyst Path\Projects\Analytical Projects\Jooble\AB Test Results\ab_test_results_repot.xlsx

Excel file created:
ab_test_results_repot.xlsx - Complete report with summary, detailed views, and visualizations
